# O que aconteceu com as empresas brasileiras em 2018? (CNPJ)

Fonte: [Cadastro Nacional da Pessoa Jurídica, Receita Federal](https://www.gov.br/receitafederal/pt-br/acesso-a-informacao/dados-abertos/cadastros) · [layout oficial dos arquivos](https://www.gov.br/receitafederal/dados/cnpj-metadados.pdf)

**Pergunta:** o salto de baixas em 2018 foi uma crise ou uma limpeza do cadastro?

**Como usar**

1. Primeiro rode tudo com `PARTES = [1]` (342 MB), só para testar.
2. Se deu certo, troque para `PARTES = list(range(10))` (cerca de 5,4 GB) e rode tudo de novo a partir da seção 1.

A seção 3 é a única demorada. Ela salva resumos pequenos na pasta `dados`. Se eles já existirem, a seção 3 só os carrega, então dá para mudar os gráficos à vontade sem reler a base.

## 1. Configuração

In [ ]:
import subprocess
import tempfile
import urllib.request
import zipfile
from base64 import b64encode
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap

PARTES = [1]                  # teste; para o resultado final: list(range(10))
MES = "2026-10"               # foto mensal do cadastro
ULTIMO_ANO_COMPLETO = 2025    # último ano inteiro coberto pela foto
COORTES = range(2005, 2025)   # anos de abertura (2005 a 2024); o MEI só existe desde 2009

# Códigos de situação: 1 nula, 2 ativa, 3 suspensa, 4 inapta, 8 baixada
INAPTA, BAIXADA = 4, 8

# Entidades que têm CNPJ mas não são empresas comuns
DIVISOES_FORA = {84, 94, 97, 99}   # governo, associações/partidos/igrejas, domésticos, organismos
CLASSES_FORA = {8112500}           # condomínios prediais

# Arquivos da amostra e da base completa têm nomes diferentes, para nunca se misturarem
COMPLETO = sorted(PARTES) == list(range(10))
SUFIXO = "" if COMPLETO else "_amostra"
PERIODO = f"{COORTES[0]}-{COORTES[-1]}"
PASTA_SAIDA = Path("dados")
ARQ_EMPRESAS = PASTA_SAIDA / f"empresas_{PERIODO}{SUFIXO}.csv"
ARQ_DIAS = PASTA_SAIDA / f"fechamentos_por_dia_{PERIODO}{SUFIXO}.csv"
ARQ_CONTAGEM = PASTA_SAIDA / f"contagem_{PERIODO}{SUFIXO}.csv"
ARQ_MOTIVOS = PASTA_SAIDA / "motivos.csv"

# Os zips e os resumos de cada parte ficam numa pasta temporária, fora do repositório e do
# OneDrive. Se mudar as exclusões acima, apague os "v3_*" dela e os CSVs de "dados".
PASTA_ZIP = Path(tempfile.gettempdir()) / "cnpj"
PASTA_ZIP.mkdir(parents=True, exist_ok=True)
PASTA_SAIDA.mkdir(exist_ok=True)

## 2. Download

A pasta é pública: o "usuário" abaixo é só o código do link de compartilhamento da Receita.

In [ ]:
BASE = f"https://arquivos.receitafederal.gov.br/public.php/webdav/{MES}"
TOKEN = "YggdBLfdninEJX9"


def tamanho_remoto(nome):
    """Tamanho do arquivo no servidor, em bytes, sem baixá-lo."""
    req = urllib.request.Request(f"{BASE}/{nome}", method="HEAD")
    req.add_header("Authorization", "Basic " + b64encode(f"{TOKEN}:".encode()).decode())
    req.add_header("X-Requested-With", "XMLHttpRequest")
    with urllib.request.urlopen(req, timeout=60) as resposta:
        return int(resposta.headers["Content-Length"])


def baixar(nome, tentativas=4):
    """Baixa o arquivo, continuando de onde parou se a conexão cair."""
    destino = PASTA_ZIP / nome
    esperado = tamanho_remoto(nome)
    for _ in range(tentativas):
        atual = destino.stat().st_size if destino.exists() else 0
        if atual == esperado:
            return destino
        if atual > esperado:
            destino.unlink()
        subprocess.run([
            "curl", "-sSLf", "-C", "-", "--retry", "3", "--connect-timeout", "30",
            "--speed-limit", "1000", "--speed-time", "60",
            "-u", f"{TOKEN}:", "-H", "X-Requested-With: XMLHttpRequest",
            "-o", str(destino), f"{BASE}/{nome}",
        ])
    if destino.exists() and destino.stat().st_size == esperado:
        return destino
    raise RuntimeError(f"Não consegui baixar {nome}")


# Tabela que traduz o código do motivo para texto
if not ARQ_MOTIVOS.exists():
    tabela = pd.read_csv(baixar("Motivos.zip"), compression="zip", sep=";", header=None,
                         encoding="latin-1", dtype=str, usecols=[0, 1], names=["codigo", "motivo"])
    tabela.to_csv(ARQ_MOTIVOS, index=False)

MOTIVOS = pd.read_csv(ARQ_MOTIVOS, dtype=str)
MOTIVOS["codigo"] = pd.to_numeric(MOTIVOS["codigo"], errors="coerce")
MOTIVOS = MOTIVOS.dropna(subset=["codigo"]).astype({"codigo": int}).set_index("codigo")["motivo"]


def nome_motivo(codigo):
    return MOTIVOS.get(codigo, f"código {codigo}").strip().capitalize()


print("Motivos carregados:", len(MOTIVOS))

## 3. Ler e resumir os dados (parte demorada)

O arquivo tem uma linha por **estabelecimento**. Ficamos só com as **matrizes**, então cada linha usada representa uma empresa. São lidas 6 das 30 colunas, nenhuma com nome, telefone ou e-mail.

São gerados dois resumos:

- **empresas:** quantas empresas há para cada combinação de ano de abertura, setor, situação, motivo e ano da situação.
- **fechamentos por dia:** quantas empresas ficaram inaptas ou baixadas em cada dia, por motivo.

In [ ]:
COLUNAS = {3: "matriz", 5: "situacao", 6: "data_situacao", 7: "motivo",
           10: "data_abertura", 11: "cnae"}
CHAVES = ["ano_abertura", "divisao", "situacao", "motivo", "ano_situacao", "anos_de_vida"]
CHAVES_DIA = ["data_situacao", "situacao", "motivo"]
ETAPAS = ["linhas lidas", "campos essenciais ausentes", "não são empresas comuns",
          "inscrição nula", "data de situação inválida", "empresas usadas"]


def contar(d, chaves):
    return d.groupby(chaves, as_index=False)["empresas"].sum()


def resumir(pedaco, contagem):
    """Transforma um pedaço do arquivo nos dois resumos."""
    d = pedaco.rename(columns=COLUNAS).apply(pd.to_numeric, errors="coerce")
    contagem["linhas lidas"] += len(d)

    d = d[d["matriz"] == 1]
    incompleta = d[["situacao", "data_abertura", "cnae"]].isna().any(axis=1)
    contagem["campos essenciais ausentes"] += int(incompleta.sum())
    d = d[~incompleta].copy()
    d["ano_abertura"] = d["data_abertura"] // 10000
    d["divisao"] = d["cnae"] // 100000
    d["motivo"] = d["motivo"].fillna(-1)
    d = d[d["ano_abertura"].isin(COORTES)]

    nao_empresa = d["divisao"].isin(DIVISOES_FORA) | d["cnae"].isin(CLASSES_FORA)
    contagem["não são empresas comuns"] += int(nao_empresa.sum())
    nula = d["situacao"] == 1
    contagem["inscrição nula"] += int((nula & ~nao_empresa).sum())
    d = d[~nao_empresa & ~nula].copy()

    # Datas AAAAMMDD: a subtração // 10000 dá os anos completos de vida
    d["anos_de_vida"] = (d["data_situacao"] - d["data_abertura"]) // 10000
    d["ano_situacao"] = d["data_situacao"] // 10000
    ativa = d["situacao"] == 2
    invalida = ~ativa & ~(d["data_situacao"] >= d["data_abertura"])
    contagem["data de situação inválida"] += int(invalida.sum())

    # Para empresas ativas, a data e o motivo da situação não interessam
    d.loc[ativa, ["anos_de_vida", "ano_situacao", "motivo"]] = -1
    d = d[~invalida].assign(empresas=1)
    contagem["empresas usadas"] += len(d)

    fechadas = d[d["situacao"].isin([INAPTA, BAIXADA])]
    return (contar(d.astype({c: int for c in CHAVES}), CHAVES),
            contar(fechadas.astype({c: int for c in CHAVES_DIA}), CHAVES_DIA))


def parcial(nome, parte):
    return PASTA_ZIP / f"v3_{nome}_{PERIODO}_{MES}_{parte}.csv"


def processar(parte):
    zip_path = baixar(f"Estabelecimentos{parte}.zip")
    contagem = dict.fromkeys(ETAPAS, 0)
    empresas, dias = [], []
    with pd.read_csv(
        zip_path, compression="zip", sep=";", header=None, encoding="latin-1",
        usecols=list(COLUNAS), dtype=str, chunksize=1_000_000, on_bad_lines="warn",
    ) as leitor:
        for pedaco in leitor:
            e, f = resumir(pedaco, contagem)
            empresas.append(e)
            dias.append(f)
    contar(pd.concat(empresas), CHAVES).to_csv(parcial("empresas", parte), index=False)
    contar(pd.concat(dias), CHAVES_DIA).to_csv(parcial("dias", parte), index=False)
    pd.DataFrame(contagem.items(), columns=["etapa", "n"]).to_csv(parcial("contagem", parte), index=False)
    zip_path.unlink()


if all(a.exists() for a in (ARQ_EMPRESAS, ARQ_DIAS, ARQ_CONTAGEM)):
    print("Resumos já existem em 'dados': só carregando.")
else:
    for parte in PARTES:
        if parcial("contagem", parte).exists():   # é o último arquivo gravado de cada parte
            print(f"Parte {parte}: já processada")
            continue
        print(f"Parte {parte}: baixando e processando...")
        processar(parte)

    def juntar(nome):
        return pd.concat([pd.read_csv(parcial(nome, p)) for p in PARTES])

    contar(juntar("empresas"), CHAVES).to_csv(ARQ_EMPRESAS, index=False)
    contar(juntar("dias"), CHAVES_DIA).to_csv(ARQ_DIAS, index=False)
    juntar("contagem").groupby("etapa", sort=False)["n"].sum().to_csv(ARQ_CONTAGEM)

dados = pd.read_csv(ARQ_EMPRESAS)
dias = pd.read_csv(ARQ_DIAS)
dias["data"] = pd.to_datetime(dias["data_situacao"].astype(str), format="%Y%m%d", errors="coerce")
display(pd.read_csv(ARQ_CONTAGEM))

## 4. Risco de baixa em cada ano

Para cada ano de abertura e cada ano do calendário:

> **risco = baixas naquele ano ÷ empresas que ainda não tinham sido baixadas no início daquele ano**

- Só conta **baixada**. As inaptas continuam no grupo, porque a Receita declara inaptidão em lotes e isso distorce as datas.
- O próprio ano de abertura fica de fora, porque as empresas não existiram nele o ano inteiro.

In [ ]:
def tabela_risco(d):
    linhas = {}
    for ano, g in d.groupby("ano_abertura"):
        baixas = g[g["situacao"] == BAIXADA].groupby("ano_situacao")["empresas"].sum()
        restantes = g["empresas"].sum() - baixas.get(ano, 0)
        linhas[ano] = {}
        for y in range(ano + 1, ULTIMO_ANO_COMPLETO + 1):
            if restantes > 0:
                linhas[ano][y] = 100 * baixas.get(y, 0) / restantes
            restantes -= baixas.get(y, 0)
    return pd.DataFrame(linhas).T.sort_index(axis=0).sort_index(axis=1)


MINIMO = 1000   # ignora anos de abertura com poucas empresas (acontece na amostra)
n_coorte = dados.groupby("ano_abertura")["empresas"].sum()
risco = tabela_risco(dados[dados["ano_abertura"].isin(n_coorte[n_coorte >= MINIMO].index)])
risco.round(2).to_csv(PASTA_SAIDA / f"risco{SUFIXO}.csv")
risco.round(1)

## 5. Testes da suspeita

1. **Motivo:** o salto de 2018 vem de um motivo específico, ligado à Receita, ou de fechamentos voluntários?
2. **Data:** as baixas se espalham pelo ano ou caem em poucos dias?

In [ ]:
ANO = 2018
baixas = dados[dados["situacao"] == BAIXADA]
por_motivo = baixas.pivot_table(index="motivo", columns="ano_situacao", values="empresas",
                                aggfunc="sum", fill_value=0)
por_motivo = por_motivo.reindex(columns=range(ANO - 2, ANO + 3), fill_value=0)
por_motivo["salto"] = por_motivo[ANO] - (por_motivo[ANO - 1] + por_motivo[ANO + 1]) / 2
MOTIVO_DESTAQUE = int(por_motivo["salto"].idxmax())

print(f"Teste 1: motivos que mais cresceram em {ANO}, em relação à média de {ANO - 1} e {ANO + 1}")
t1 = por_motivo.sort_values("salto", ascending=False).head(6).astype(int)
t1.insert(0, "descrição", [nome_motivo(m) for m in t1.index])
display(t1)
t1.to_csv(PASTA_SAIDA / f"teste_motivos{SUFIXO}.csv")

print(f"Teste 2: dias com mais baixas em {ANO}")
do_ano = dias[(dias["situacao"] == BAIXADA) & (dias["data"].dt.year == ANO)]
top = do_ano.groupby("data")["empresas"].sum().sort_values(ascending=False)
t2 = pd.DataFrame({"baixas": top, "% do ano": 100 * top / top.sum()}).head(10)
t2["% acumulada"] = t2["% do ano"].cumsum()
display(t2.round(1))
t2.round(2).to_csv(PASTA_SAIDA / f"teste_dias{SUFIXO}.csv")

print("Comparação: % das baixas de cada ano que caíram nos 5 dias mais movimentados")
t3 = {}
for y in range(2013, ULTIMO_ANO_COMPLETO + 1):
    s = dias[(dias["situacao"] == BAIXADA) & (dias["data"].dt.year == y)].groupby("data")["empresas"].sum()
    t3[y] = 100 * s.nlargest(5).sum() / s.sum() if len(s) else np.nan
t3 = pd.Series(t3, name="% nos 5 maiores dias")
display(t3.round(1).to_frame().T)
t3.round(2).to_csv(PASTA_SAIDA / f"teste_concentracao{SUFIXO}.csv")

print(f"Teste 3: % das empresas de cada ano de abertura baixadas em {ANO} pelo motivo "
      f"'{nome_motivo(MOTIVO_DESTAQUE)}'")
alvo = baixas[(baixas["ano_situacao"] == ANO) & (baixas["motivo"] == MOTIVO_DESTAQUE)]
t4 = (100 * alvo.groupby("ano_abertura")["empresas"].sum() / n_coorte).fillna(0).rename("%")
display(t4[n_coorte >= MINIMO].round(1).to_frame().T)

print("Teste 4: cancelamentos em 01/02/2018 (a Receita anunciou 1.372.246 CNPJs de MEI baixados)")
cancelados = dias[(dias["data_situacao"] == 20180201) & (dias["situacao"] == BAIXADA)
                  & (dias["motivo"] == MOTIVO_DESTAQUE)]["empresas"].sum()
print(f"{cancelados:,} empresas nesta base".replace(",", "."))

## 6. Gráfico principal: ano de abertura × ano do calendário

In [ ]:
# Troque o título e o subtítulo pela SUA conclusão depois de olhar os dados
TITULO_A = "Em que ano as empresas foram baixadas?"
SUBTITULO_A = "% das empresas ainda não baixadas que foram baixadas em cada ano. Mais escuro = mais baixas."
TITULO_B = "Baixas de empresas por mês"
SUBTITULO_B = f"Empresas abertas a partir de {COORTES[0]}, pelo mês em que foram baixadas."

AZUIS = LinearSegmentedColormap.from_list(
    "azuis", ["#e8f1fd", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
FUNDO, TINTA, TINTA2, GRADE, CINZA = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({"figure.facecolor": FUNDO, "axes.facecolor": FUNDO, "font.size": 11,
                     "text.color": TINTA, "axes.labelcolor": TINTA2,
                     "xtick.color": TINTA2, "ytick.color": TINTA2})

RODAPE = ("" if COMPLETO else "AMOSTRA, NÃO USAR NA ENTREGA. ") + (
    f"Fonte: Receita Federal, CNPJ ({MES}). Só matrizes; sem governo, associações e condomínios.")


def cabecalho(fig, titulo, subtitulo):
    fig.text(0.02, 0.97, titulo, fontsize=17, fontweight="bold", va="top")
    fig.text(0.02, 0.92, subtitulo, fontsize=11.5, color=TINTA2, va="top")
    fig.text(0.02, 0.015, RODAPE, fontsize=9, color=TINTA2)


def grafico_risco(t):
    fig, ax = plt.subplots(figsize=(14, 10))
    fig.subplots_adjust(left=0.08, right=0.9, top=0.83, bottom=0.12)
    vmax = np.nanmax(t.values)
    ax.pcolormesh(np.ma.masked_invalid(t.values), cmap=AZUIS, vmin=0, vmax=vmax,
                  edgecolors=FUNDO, linewidth=2)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.iat[i, j]
            if not np.isnan(v):
                ax.text(j + 0.5, i + 0.5, f"{v:.0f}", ha="center", va="center", fontsize=10,
                        color="white" if v / vmax > 0.6 else TINTA)
    for i, ano in enumerate(t.index):
        ax.text(t.shape[1] + 0.25, i + 0.5, f"{n_coorte[ano] / 1000:,.0f} mil".replace(",", "."),
                va="center", fontsize=9.5, color=TINTA2)
    ax.text(t.shape[1] + 0.25, -0.25, "Abertas", fontsize=9.5, color=TINTA2)
    ax.set_xticks(np.arange(t.shape[1]) + 0.5, t.columns)
    ax.set_yticks(np.arange(t.shape[0]) + 0.5, t.index)
    for rotulo, ano in zip(ax.get_xticklabels(), t.columns):
        if ano == ANO:
            rotulo.set(fontweight="bold", color=TINTA)
    ax.set_xlabel("Ano do calendário", labelpad=8)
    ax.set_ylabel("Ano de abertura", labelpad=8)
    ax.invert_yaxis()
    ax.tick_params(length=0)
    ax.spines[:].set_visible(False)
    cabecalho(fig, TITULO_A, SUBTITULO_A)
    return fig


fig = grafico_risco(risco)
fig.savefig(PASTA_SAIDA / f"vis-risco{SUFIXO}.png", dpi=200)
plt.show()

## 7. Gráfico de apoio: baixas por mês

Destaca o motivo que mais cresceu em 2018.

In [ ]:
def grafico_meses():
    d = dias[(dias["situacao"] == BAIXADA) & (dias["data"].dt.year <= ULTIMO_ANO_COMPLETO)]
    mes = d["data"].dt.to_period("M").dt.to_timestamp()
    total = d.groupby(mes)["empresas"].sum() / 1000
    destaque = d[d["motivo"] == MOTIVO_DESTAQUE].groupby(mes)["empresas"].sum() / 1000
    destaque = destaque.reindex(total.index, fill_value=0)

    fig, ax = plt.subplots(figsize=(12, 6.5))
    fig.subplots_adjust(left=0.07, right=0.97, top=0.77, bottom=0.12)
    ax.bar(total.index, total - destaque, width=24, color=CINZA, label="Demais motivos")
    ax.bar(total.index, destaque, bottom=total - destaque, width=24, color="#184f95",
           label=nome_motivo(MOTIVO_DESTAQUE))
    ax.set_ylabel("Mil empresas baixadas no mês", labelpad=8)
    ax.grid(axis="y", color=GRADE, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    ax.spines[:].set_visible(False)
    ax.legend(loc="lower left", bbox_to_anchor=(-0.01, 1.0), ncols=2, frameon=False)
    cabecalho(fig, TITULO_B, SUBTITULO_B)
    return fig


fig = grafico_meses()
fig.savefig(PASTA_SAIDA / f"vis-meses{SUFIXO}.png", dpi=200)
plt.show()

## 8. Salvar os resultados

No Colab, esta célula baixa um zip com as tabelas e figuras. Coloque a pasta `dados/` no repositório junto com este notebook.

In [ ]:
try:
    import shutil

    from google.colab import files
    files.download(shutil.make_archive("resultados", "zip", ".", "dados"))
except ImportError:
    print("Resultados em", PASTA_SAIDA.resolve())